# Xuất UniBot vòng 2 sang bản nén q4 (chạy nhanh, nhẹ RAM)

Không huấn luyện lại: nạp mô hình gốc Qwen2.5-3B-Instruct cùng LoRA vòng 2 (`unibot-lora-v2.zip`), rồi xuất GGUF nén **q4_k_m** (khoảng 1,9 GB thay vì 3,3 GB của bản q8_0).

**Trước khi chạy:** *Thời gian chạy → Thay đổi loại thời gian chạy → T4 GPU* là đủ (L4 cũng được). Chạy lần lượt từng ô (Shift + Enter). Tổng cộng khoảng 10–15 phút.

Built with Qwen. Mô hình gốc dùng Qwen Research License (phi thương mại).

In [ ]:
%%capture
!pip install unsloth

## 1. Tải file LoRA vòng 2 lên
Chọn `unibot-lora-v2.zip` trong thư mục `training` của dự án.

In [ ]:
import glob, os, zipfile
from google.colab import files

uploaded = files.upload()
name = next(n for n in uploaded if n.endswith(".zip"))
with zipfile.ZipFile(name) as z:
    z.extractall("unibot-lora")
found = glob.glob("unibot-lora/**/adapter_config.json", recursive=True)
assert found, "File zip không đúng bản LoRA"
lora_dir = os.path.dirname(found[0])
print("LoRA:", lora_dir, sorted(os.listdir(lora_dir)))


## 2. Nạp mô hình gốc kèm LoRA và xuất q4_k_m

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(model_name=lora_dir, max_seq_length=3072, load_in_4bit=False)
model.save_pretrained_gguf("unibot-q4", tokenizer, quantization_method="q4_k_m")
gguf = sorted(glob.glob("unibot-q4*/*.gguf") + glob.glob("*.gguf"))
print(gguf)

## 3. Tải file về máy
Nếu tải bị treo, dùng ô 4 để chép qua Google Drive rồi tải từ Drive.

In [ ]:
files.download([g for g in gguf if "q4" in g.lower()][-1])

In [ ]:
import shutil
from google.colab import drive

drive.mount("/content/drive")
src = [g for g in gguf if "q4" in g.lower()][-1]
shutil.copy(src, "/content/drive/MyDrive/unibot-v2-q4.gguf")
print("Đã chép vào Google Drive: unibot-v2-q4.gguf")